In [ ]:
import pathlib as Path
import numpy as np
import pandas as pd


## Create dataframe from embeddings

In [ ]:
#-----Frequency Embeddings dataframe creation-----
def create_embeddings_dataframe(root_path):
    emb_root = Path.Path(root_path)
    emb_files = emb_root.rglob("embeddings_samples.npz")
    
    all_data = []
    
    for emb_file in emb_files:
        emb = np.load(emb_file)
        X = emb['X']
        y = emb['y']
        subjects = emb['subs']
        
        for i in range(X.shape[0]):
            data_point = {
                'embedding': X[i],
                'label': y[i],
                'subject': subjects[i],
                'file_path': str(emb_file)
            }
            all_data.append(data_point)
    
    df = pd.DataFrame(all_data)
    return df


## Clean embedding DF

In [ ]:

def df_cleaning(df, emb_size=384):
        # --- Expand embedding column into 381 separate columns ---
    embedding_df = pd.DataFrame(df["embedding"].tolist(),
                                columns=[f"emb_{i}" for i in range(emb_size)])

    # --- Concatenate back to the original DataFrame (optional) ---
    df_expanded = pd.concat([df.drop(columns=["embedding"]), embedding_df], axis=1)
    return df_expanded

In [ ]:
#-----Frequency Embeddings dataframe creation-----
freq_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//frequency_emb_stored//")
#-----Temporal Embeddings dataframe creation-----
temp_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//temporal_emb_stored//")
#-----Combined Embeddings dataframe creation-----
comb_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//emb_stored//")

freq_emb_df= df_cleaning(freq_emb, emb_size=384)
temp_emb_df= df_cleaning(temp_emb, emb_size=384)
comb_emb_df= df_cleaning(comb_emb, emb_size=768)

In [ ]:

metadata_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_metadata//preDLB_shared(PSY_RAW).csv')
df_metadata = pd.read_csv(metadata_path, sep=";", encoding="utf-8-sig")

In [ ]:
clinical_data_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_metadata//clinical_data_csv.csv')
df_clinical = pd.read_csv(clinical_data_path, sep=",", encoding="utf-8-sig")

In [ ]:
handcrafted_features_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_hf//corpus_LBD_CZ_002_writing_results_table_original_filtered_extended.csv')
df_handcrafted_features = pd.read_csv(handcrafted_features_path, sep=";", encoding="utf-8-sig")

In [ ]:
import re
import pandas as pd

def append_col_when_main_contains_source(
    df_main, df_source, *, 
    match_col_main="subject",            # in df_main
    match_col_source="ID_1.meranie",     # in df_source
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
):
    df_main = df_main.copy()
    df_source = df_source.copy()

    # normalize + keep only rows in source with non-missing values
    df_main[match_col_main] = df_main[match_col_main].astype(str).str.strip()
    df_source[match_col_source] = df_source[match_col_source].astype(str).str.strip()
    df_source = df_source.dropna(subset=[value_col])

    # init target column
    if new_col_name not in df_main:
        df_main[new_col_name] = pd.NA

    # for each source row, mark all main rows whose subject CONTAINS the source token
    for _, r in df_source.iterrows():
        token = r[match_col_source]
        if not token:
            continue
        mask = df_main[match_col_main].str.contains(re.escape(token), na=False, case=not case)
        # write only where we don't have a value yet (keeps first hit)
        to_set = mask & df_main[new_col_name].isna()
        df_main.loc[to_set, new_col_name] = r[value_col]

    return df_main

In [ ]:
freq_emb_df_lbl = append_col_when_main_contains_source(
    df_main=freq_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [ ]:
freq_emb_df_lbl.to_csv("LBD_CZ_002_COBEN_dfs/freq_emb_df_half_lbl.csv", sep=";")

In [ ]:

temp_emb_df_lbl = append_col_when_main_contains_source(
    df_main=temp_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [ ]:

temp_emb_df_lbl.to_csv("LBD_CZ_002_COBEN_dfs/temp_emb_df_half__lbl.csv", sep=";")

In [ ]:

comb_emb_df_lbl = append_col_when_main_contains_source(
    df_main=comb_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [ ]:
comb_emb_df_lbl.to_csv("LBD_CZ_002_COBEN_dfs/comb_emb_df_half__lbl.csv", sep=";")

In [ ]:

df_handcrafted_lbl = append_col_when_main_contains_source(
    df_main=df_handcrafted_features,
    df_source=df_metadata,
    match_col_main="ID",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)


In [ ]:

df_handcrafted_lbl.to_csv("LBD_CZ_002_COBEN_dfs/df_handcrafted_half_lbl.csv", sep=";")

In [ ]:
#temp_emb_df_complet_lbl = pd.read_csv("LBD_CZ_002_COBEN_dfs/temp_emb_df_lbl.csv", sep=";")

#freq_emb_df_complet_lbl= pd.read_csv("./LBD_CZ_002_COBEN_dfs/freq_emb_df_lbl.csv", sep=";")

#comb_emb_df_complet_lbl= pd.read_csv("LBD_CZ_002_COBEN_dfs/comb_emb_df_lbl.csv", sep=";")

hf_emb_df_complet_lbl= pd.read_csv("LBD_CZ_002_COBEN_dfs/df_handcrafted_lbl.csv", sep=";")


In [ ]:

import pandas as pd

def assign_diagnosis(df: pd.DataFrame,
                     subject_col: str = "subject",
                     diagnosis_col: str = "diagnosis") -> pd.DataFrame:
    """Populate `diagnosis_col` based on text in `subject_col`.

    - rows whose subject contains '#' are left untouched
    - 'HC' → 0.0, 'AD' → 2.0, 'PD' → 4.0
    """
    if diagnosis_col not in df.columns:
        df[diagnosis_col] = pd.NA

    subj = df[subject_col].astype(str)
    has_hash = subj.str.contains("#", na=False)

    mapping = {"HC": 0.0, "AD": 2.0, "PD": 4.0}
    for key, value in mapping.items():
        mask = (~has_hash) & subj.str.contains(key, na=False)
        df.loc[mask, diagnosis_col] = value

    return df


In [ ]:
temp_emb_df_complet_lbl = assign_diagnosis(temp_emb_df_lbl)

freq_emb_df_complet_lbl= assign_diagnosis(freq_emb_df_lbl)

comb_emb_df_complet_lbl=assign_diagnosis(comb_emb_df_lbl)


In [ ]:
import pandas as pd

def assign_diagnosis(df: pd.DataFrame,
                     subject_col: str = "subject",
                     diagnosis_col: str = "diagnosis") -> pd.DataFrame:
    """Populate `diagnosis_col` based on text in `subject_col`.

    - rows whose subject contains '#' are left untouched
    - 'HC' → 0.0, 'AD' → 2.0, 'PD' → 4.0
    """
    if diagnosis_col not in df.columns:
        df[diagnosis_col] = pd.NA

    subj = df[subject_col].astype(str)
    has_hash = subj.str.contains("#", na=False)

    mapping = {"HC": 0.0, "AD": 2.0, "PD": 4.0}
    for key, value in mapping.items():
        mask = (~has_hash) & subj.str.contains(key, na=False)
        df.loc[mask, diagnosis_col] = value

    return df


In [ ]:
def append_multiple_cols_when_main_contains_source(
    df_main, df_source, *,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_cols=("HC0_nHC1_MCI2_MCILB3_baseline",),
    case=False
):
    df_main = df_main.copy()
    df_source = df_source.copy()

    # normalize + clean
    df_main[match_col_main] = df_main[match_col_main].astype(str).str.strip()
    df_source[match_col_source] = df_source[match_col_source].astype(str).str.strip()

    # initialize missing columns
    for col in value_cols:
        if col not in df_main.columns:
            df_main[col] = pd.NA

    # iterate through source
    for _, r in df_source.iterrows():
        token = r[match_col_source]
        if not token or pd.isna(token):
            continue

        # use 'case' argument as passed (was inverted before)
        mask = df_main[match_col_main].str.contains(re.escape(str(token)), na=False, case=case)
        to_set = mask

        # assign all defined value columns
        for col in value_cols:
            if col not in r or pd.isna(r[col]):
                continue
            df_main.loc[to_set & df_main[col].isna(), col] = r[col]

    return df_main


In [ ]:
clinical_hf_emb_df_complet_lbl = append_multiple_cols_when_main_contains_source(
    df_main=hf_emb_df_complet_lbl,
    df_source=df_clinical,
    match_col_main="ID",
    match_col_source="#personalID",
    value_cols=["age",
                "gender",
                "MOCA",
                "education type",
                "education length",
                "memory z-score",
                "visuo-spatial z-score",
                "attention z-score",
                "executive function z-score",
                "GDS"]
)

In [ ]:
temp_emb_df_complet_lbl_2 = temp_emb_df_complet_lbl.drop(temp_emb_df_complet_lbl.columns[0], axis=1)
freq_emb_df_complet_lbl_2 = freq_emb_df_complet_lbl.drop(freq_emb_df_complet_lbl.columns[0], axis=1)
comb_emb_df_complet_lbl_2 = comb_emb_df_complet_lbl.drop(comb_emb_df_complet_lbl.columns[0], axis=1)
hf_emb_df_complet_lbl_2 = hf_emb_df_complet_lbl.drop(hf_emb_df_complet_lbl.columns[0], axis=1)

In [ ]:

hf_emb_df_complet_lbl = hf_emb_df_complet_lbl.drop(hf_emb_df_complet_lbl.columns[0], axis=1)

# Spearsman correlation / finding covariants

In [ ]:

cols_to_convert = clinical_hf_emb_df_complet_lbl.columns[1:]

clinical_hf_emb_df_complet_lbl[cols_to_convert] = (
    clinical_hf_emb_df_complet_lbl[cols_to_convert]
    .apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', '.'), errors='coerce'))
)

In [ ]:
clinical_hf_emb_df_complet_lbl.dtypes

In [ ]:

from scipy import stats
import numpy as np

meta_cols=["age",
         #   "gender",
            "MOCA",
            "education type",
            "education length",
            "memory z-score",
            "visuo-spatial z-score",
            "attention z-score",
            "executive function z-score",
            "GDS"]

feature_cols = clinical_hf_emb_df_complet_lbl.drop(columns=['ID', 'diagnosis','gender'] + meta_cols).columns.tolist()


def correlation_to_df(df, feature_cols, meta_cols, corr_type='spearman'):
    from scipy import stats
    corr_results = []
    for feature in feature_cols:
        for meta in meta_cols:
            feature_data = pd.to_numeric(df[feature], errors='coerce')
            meta_data = pd.to_numeric(df[meta], errors='coerce')
            if corr_type == 'spearman':
                corr, p_value = stats.spearmanr(feature_data, meta_data, nan_policy='omit')
            elif corr_type == 'pearson':
                corr, p_value = stats.pearsonr(feature_data.dropna(), meta_data.dropna())
            else:
                raise ValueError("corr_type must be 'spearman' or 'pearson'")
            corr_results.append({
                'feature': feature,
                'meta_variable': meta,
                'correlation': corr,
                'p_value': p_value
            })
    corr_df = pd.DataFrame(corr_results)
    return corr_df


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

def safe_spearman(x, y, min_n=3):
    """
    x, y: 1D array-like
    returns (rho, p, n_used)
    """
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    y = pd.to_numeric(pd.Series(y), errors="coerce")

    mask = x.notna() & y.notna()
    x2 = x[mask].values
    y2 = y[mask].values
    n = len(x2)

    if n < min_n:
        return np.nan, np.nan, n

    # constant vectors -> undefined correlation
    if np.all(x2 == x2[0]) or np.all(y2 == y2[0]):
        return np.nan, np.nan, n

    rho, p = stats.spearmanr(x2, y2)
    # spearmanr can still return nan if ties/degenerate
    if not np.isfinite(rho):
        rho, p = np.nan, np.nan
    return rho, p, n


def correlation_to_df(df, feature_cols, meta_cols, corr_type="spearman", min_n=3):
    rows = []
    for feat in feature_cols:
        feature_data = df[feat]

        for meta in meta_cols:
            meta_data = df[meta]

            if corr_type == "spearman":
                corr, p_value, n_used = safe_spearman(feature_data, meta_data, min_n=min_n)
            elif corr_type == "pearson":
                # paired dropna for pearson too
                x = pd.to_numeric(feature_data, errors="coerce")
                y = pd.to_numeric(meta_data, errors="coerce")
                mask = x.notna() & y.notna()
                x2, y2 = x[mask].values, y[mask].values
                n_used = len(x2)
                if n_used < min_n or np.all(x2 == x2[0]) or np.all(y2 == y2[0]):
                    corr, p_value = np.nan, np.nan
                else:
                    corr, p_value = stats.pearsonr(x2, y2)
            else:
                raise ValueError("corr_type must be 'spearman' or 'pearson'")

            rows.append({
                "feature": feat,
                "meta": meta,
                "corr": corr,
                "p_value": p_value,
                "n_used": n_used
            })

    return pd.DataFrame(rows)


In [ ]:
clinical_hf_emb_df_complet_lbl

In [ ]:
df_hf_clinical_corr = correlation_to_df(clinical_hf_emb_df_complet_lbl, feature_cols, meta_cols, corr_type='spearman')

In [ ]:
df_hf_clinical_corr

In [ ]:
df_hf_clinical_corr = df_hf_clinical_corr.drop(df_hf_clinical_corr.columns[-1], axis=1) 

In [ ]:
df_hf_clinical_corr

In [ ]:
# Create separate pivots for correlation and p-value
#df_corr_pivot = df_hf_clinical_corr.pivot(
#    index='feature',
#    columns='meta',
#    values='corr'
#)

df_pval_pivot = df_hf_clinical_corr.pivot(
    index='feature',
    columns='meta',
    values='p_value'
)

In [ ]:
df_pval_pivot[df_pval_pivot < 0.05].count()

# FDR correction

In [ ]:

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

def fdr_correction_matrix(df_pvals: pd.DataFrame, alpha: float = 0.05, method: str = "fdr_bh"):
    df = df_pvals.copy()

    # Only numeric columns (skip "feature")
    numeric_cols = df.select_dtypes(include=[float, int]).columns
    arr = df[numeric_cols].to_numpy(dtype=float)

    # Mask NaNs
    mask_na = np.isnan(arr)

    # Take only valid p-values (flatten)
    pvals_flat = arr[~mask_na]

    # FDR
    reject, pvals_corr, _, _ = multipletests(pvals_flat, alpha=alpha, method=method)

    # Prepare output arrays
    arr_fdr = np.full_like(arr, np.nan, dtype=float)
    arr_sig = np.zeros_like(arr, dtype=bool)

    # Fill corrected values
    arr_fdr[~mask_na] = pvals_corr
    arr_sig[~mask_na] = reject

    # Back to DataFrames
    df_fdr = df.copy()
    df_sig = df.copy()

    df_fdr[numeric_cols] = arr_fdr
    df_sig[numeric_cols] = arr_sig

    return df_fdr, df_sig


In [ ]:

df_fdr, df_sig = fdr_correction_matrix(df_pval_pivot, alpha=0.05)

#Show significant (feature, metadata) pairs
significant_pairs = (
    df_sig
         .iloc[:, 1:]         # skip the 'feature' column
         .stack()             # long format
         .loc[lambda s: s]    # keep only True
         .index
         .to_frame(name=["feature", "metadata"])
)
##
significant_pairs


In [ ]:

df_after_fdr = df_fdr[df_sig]

In [ ]:

df_after_fdr.dropna(how='all', inplace=True)
df_after_fdr

# Task-wise Dataframe dictionaries creation

- temp_emb_df_complet_lbl_2
- freq_emb_df_complet_lbl_2
- comb_emb_df_complet_lbl_2
- hf_emb_df_complet_lbl_2 

In [ ]:
def task_wise_df_creation(df, task_name):
    task_df = df[df['file_path'].str.contains(task_name)].reset_index(drop=True)
    return task_df

In [ ]:

task_list = ['1_1', '9_1', '15_1', '16_1', '17_1', '18_1', '19_1']

task_freq_dfs = {task: task_wise_df_creation(freq_emb_df_complet_lbl, task) for task in task_list}
task_temp_dfs = {task: task_wise_df_creation(temp_emb_df_complet_lbl, task) for task in task_list}
task_comb_dfs = {task: task_wise_df_creation(comb_emb_df_complet_lbl, task) for task in task_list}

In [ ]:

task_list = ['1_1', '9_1', '15_1', '16_1', '17_1', '18_1', '19_1']

task_hf_dfs = {
    task: hf_emb_df_complet_lbl.loc[
        :,
        hf_emb_df_complet_lbl.columns.astype(str).str.contains(task, case=False) |
        hf_emb_df_complet_lbl.columns.isin(['diagnosis', 'ID'])
    ].copy()
    for task in task_list
}

## MCI-LBD vs HC

In [ ]:

task_freq_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2, 4])].reset_index(drop=True) for k, v in task_freq_dfs.items()}
task_temp_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2, 4])].reset_index(drop=True) for k, v in task_temp_dfs.items()}
task_comb_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2, 4])].reset_index(drop=True) for k, v in task_comb_dfs.items()}
task_hf_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2, 4])].reset_index(drop=True) for k, v in task_hf_dfs.items()}

## MCI-AD vs HC

In [ ]:
task_freq_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3, 4])].reset_index(drop=True) for k, v in task_freq_dfs.items()}
task_temp_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3, 4])].reset_index(drop=True) for k, v in task_temp_dfs.items()}
task_comb_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3, 4])].reset_index(drop=True) for k, v in task_comb_dfs.items()}
task_hf_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3, 4])].reset_index(drop=True) for k, v in task_hf_dfs.items()}

## MCI-PD vs HC

In [ ]:

task_freq_dfs_pd_hc = {k: v[~v["diagnosis"].isin([1, 3, 2])].reset_index(drop=True) for k, v in task_freq_dfs.items()}
task_temp_dfs_pd_hc = {k: v[~v["diagnosis"].isin([1, 3, 2])].reset_index(drop=True) for k, v in task_temp_dfs.items()}
task_comb_dfs_pd_hc = {k: v[~v["diagnosis"].isin([1, 3, 2])].reset_index(drop=True) for k, v in task_comb_dfs.items()}
task_hf_dfs_pd_hc = {k: v[~v["diagnosis"].isin([1, 3, 2])].reset_index(drop=True) for k, v in task_hf_dfs.items()}

## MCI-LBD vs MCI-AD

In [ ]:

task_freq_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0, 4])].reset_index(drop=True) for k, v in task_freq_dfs.items()}
task_temp_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0, 4])].reset_index(drop=True) for k, v in task_temp_dfs.items()}
task_comb_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0, 4])].reset_index(drop=True) for k, v in task_comb_dfs.items()}
task_hf_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0, 4])].reset_index(drop=True) for k, v in task_hf_dfs.items()}

In [ ]:
task_freq_dfs_lbd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_freq_dfs_lbd_hc.items()}
task_temp_dfs_lbd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_temp_dfs_lbd_hc.items()}
task_comb_dfs_lbd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_comb_dfs_lbd_hc.items()}

In [ ]:

task_freq_dfs_ad_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_freq_dfs_ad_hc.items()}
task_temp_dfs_ad_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_temp_dfs_ad_hc.items()}
task_comb_dfs_ad_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_comb_dfs_ad_hc.items()}

In [ ]:
task_freq_dfs_pd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_freq_dfs_pd_hc.items()}
task_temp_dfs_pd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_temp_dfs_pd_hc.items()}
task_comb_dfs_pd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_comb_dfs_pd_hc.items()}

In [ ]:

task_freq_dfs_ad_lbd = {k: v.drop(columns=['label', 'file_path']) for k, v in task_freq_dfs_ad_lbd.items()}
task_temp_dfs_ad_lbd = {k: v.drop(columns=['label', 'file_path']) for k, v in task_temp_dfs_ad_lbd.items()}
task_comb_dfs_ad_lbd = {k: v.drop(columns=['label', 'file_path']) for k, v in task_comb_dfs_ad_lbd.items()}

In [ ]:
for dfs in (task_freq_dfs_lbd_hc, task_temp_dfs_lbd_hc, task_comb_dfs_lbd_hc, task_hf_dfs_lbd_hc):
    
    for k, v in dfs.items():
        if "diagnosis" in v.columns:
            v["diagnosis"] = v["diagnosis"].replace({3.0: 1, 0.0: 0})


In [ ]:
for dfs in (task_freq_dfs_ad_hc, task_temp_dfs_ad_hc, task_comb_dfs_ad_hc, task_hf_dfs_ad_hc):
    
    for k, v in dfs.items():
        if "diagnosis" in v.columns:
            v["diagnosis"] = v["diagnosis"].replace({2.0: 1, 0.0: 0})


In [ ]:

for dfs in (task_freq_dfs_ad_lbd, task_temp_dfs_ad_lbd, task_comb_dfs_ad_lbd, task_hf_dfs_ad_lbd):
    
    for k, v in dfs.items():
        if "diagnosis" in v.columns:
            v["diagnosis"] = v["diagnosis"].replace({3.0: 1, 2.0: 0})

In [ ]:
for dfs in (task_freq_dfs_pd_hc, task_temp_dfs_pd_hc, task_comb_dfs_pd_hc, task_hf_dfs_pd_hc):
    
    for k, v in dfs.items():
        if "diagnosis" in v.columns:
            v["diagnosis"] = v["diagnosis"].replace({4.0: 1, 0.0: 0})


In [ ]:
task_freq_dfs_ad_hc.get("1_1")

In [ ]:

def miss_val_clean(df, percent_threshold=0.8):
    """
    Cleans the DataFrame by removing columns with more than the specified percentage of missing values.
    
    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame to be cleaned.
    percent_threshold : float
        The maximum allowed percentage of missing values in a column (between 0 and 1).
        
    Returns
    -------
    pd.DataFrame
        The cleaned DataFrame with columns exceeding the missing value threshold removed.
    """
    df_cleaned = df.copy()
    df_cleaned = df_cleaned.loc[:, df_cleaned.isnull().mean() <= percent_threshold]
    
    cols_to_convert = df_cleaned.columns[1:-1]

    df_cleaned[cols_to_convert] = (
        df_cleaned[cols_to_convert]
        .apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', '.'), errors='coerce'))
    )

    df_cleaned = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))
    return df_cleaned

In [ ]:

task_hf_dfs_clean_lbd_hc = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_hf_dfs_lbd_hc.items()}
task_hf_dfs_clean_ad_hc = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_hf_dfs_ad_hc.items()}
task_hf_dfs_clean_ad_lbd = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_hf_dfs_ad_lbd.items()}
task_hf_dfs_clean_pd_hc = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_hf_dfs_pd_hc.items()}

In [ ]:

# Use this one 
def merge_on_subject_agg_right(df1, df2):
    df1 = df1.copy(); df2 = df2.copy()
    if 'subject' not in df1 and 'ID' in df1: df1 = df1.rename(columns={'ID':'subject'})
    if 'subject' not in df2 and 'ID' in df2: df2 = df2.rename(columns={'ID':'subject'})
    num_cols = df2.select_dtypes('number').columns.tolist()
    agg = {c:'first' for c in df2.columns if c not in num_cols and c!='subject'}
    agg.update({c:'mean' for c in num_cols})
    df2 = df2.groupby('subject', as_index=False).agg(agg)
    return pd.merge(df1, df2, on='subject', how='inner', validate='many_to_one', sort=False)


### MCI-LBD vs HC

In [ ]:
task_hf_compensated_freq_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_freq_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

In [ ]:

task_hf_compensated_temp_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_temp_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

In [ ]:

task_hf_compensated_comb_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_comb_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_comb_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

In [ ]:
task_hf_comb_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_comb_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_comb_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

In [ ]:
task_hf_freq_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_freq_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

In [ ]:
task_hf_temp_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_temp_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

### MCI-AD vs HC

In [ ]:
### MCI-AD vs HC
task_hf_compensated_freq_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_freq_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}

task_hf_compensated_temp_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_temp_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}

task_hf_compensated_comb_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_comb_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_comb_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}
task_hf_comb_emb_lbl_ad_hc= {
    k: merge_on_subject_agg_right(task_comb_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_comb_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}
task_hf_freq_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_freq_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}
task_hf_temp_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_temp_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}

In [ ]:
task_hf_compensated_freq_emb_lbl_ad_hc.get("1_1")

### MCI-LBD vs MCI-AD

In [ ]:
### MCI-LBD vs MCI-LBD
task_hf_compensated_freq_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_freq_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_freq_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}

task_hf_compensated_temp_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_temp_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_temp_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}

task_hf_compensated_comb_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_comb_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_comb_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}
task_hf_comb_emb_lbl_ad_lbd= {
    k: merge_on_subject_agg_right(task_comb_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_comb_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}
task_hf_freq_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_freq_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_freq_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}
task_hf_temp_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_temp_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_temp_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}

### PD vs HC

In [ ]:

### MCI-AD vs HC
task_hf_compensated_freq_emb_lbl_pd_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_pd_hc[k], task_hf_dfs_clean_pd_hc[k])
    for k in task_freq_dfs_pd_hc.keys() & task_hf_dfs_clean_pd_hc.keys()
}

task_hf_compensated_temp_emb_lbl_pd_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_pd_hc[k], task_hf_dfs_clean_pd_hc[k])
    for k in task_temp_dfs_pd_hc.keys() & task_hf_dfs_clean_pd_hc.keys()
}

task_hf_compensated_comb_emb_lbl_pd_hc = {
    k: merge_on_subject_agg_right(task_comb_dfs_pd_hc[k], task_hf_dfs_clean_pd_hc[k])
    for k in task_comb_dfs_pd_hc.keys() & task_hf_dfs_clean_pd_hc.keys()
}
task_hf_comp_emb_lbl_pd_hc= {
    k: merge_on_subject_agg_right(task_comb_dfs_pd_hc[k], task_hf_dfs_clean_pd_hc[k])
    for k in task_comb_dfs_pd_hc.keys() & task_hf_dfs_clean_pd_hc.keys()
}
task_hf_freq_emb_lbl_pd_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_pd_hc[k], task_hf_dfs_clean_pd_hc[k])
    for k in task_freq_dfs_pd_hc.keys() & task_hf_dfs_clean_pd_hc.keys()
}
task_hf_temp_emb_lbl_pd_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_pd_hc[k], task_hf_dfs_clean_pd_hc[k])
    for k in task_temp_dfs_pd_hc.keys() & task_hf_dfs_clean_pd_hc.keys()
}

In [ ]:
def x_y_split(df, split_param):
    if split_param is None or split_param == "diagnosis_y":
        #embedding_cols = df[1:-1]
        X = df.iloc[:,1:-2].values
        y = df["diagnosis_y"].values
    elif split_param != None: 
        embedding_cols = [c for c in df.columns if c.startswith(split_param)]
        #X = df[embedding_cols].values
        X = df.iloc[:,1:-2].values
        y = df["diagnosis"].values
    
    return X, y



In [ ]:
def remove_leaking_labels(df, label_string = 'diagnosis_x'):
    df_cleaned = df.copy()
    df_cleaned = df_cleaned.drop(label_string, axis = 1)
    return df_cleaned

In [ ]:
task_hf_compensated_freq_emb_lbl_ad_hc.get("1_1")

In [ ]:
task_hf_compensated_freq_emb_lbl_ad_hc.get("1_1")

In [ ]:
task_hf_compensated_freq_emb_lbl_ad_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_hc.items()}
task_hf_compensated_temp_emb_lbl_ad_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_hc.items()}
task_hf_compensated_comb_emb_lbl_ad_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_hc.items()}

task_hf_compensated_freq_emb_lbl_lbd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_lbd_hc.items()}
task_hf_compensated_temp_emb_lbl_lbd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_lbd_hc.items()}
task_hf_compensated_comb_emb_lbl_lbd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_lbd_hc.items()}


task_hf_compensated_freq_emb_lbl_ad_lbl = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_lbd.items()}
task_hf_compensated_temp_emb_lbl_ad_lbd = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_lbd.items()}
task_hf_compensated_comb_emb_lbl_ad_lbd = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_lbd.items()}

task_hf_compensated_freq_emb_lbl_pd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_pd_hc.items()}
task_hf_compensated_temp_emb_lbl_pd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_pd_hc.items()}
task_hf_compensated_comb_emb_lbl_pd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_pd_hc.items()}

In [ ]:
def leaking_sanity_test(df, label_string='diagnosis_x'):
    if label_string in df.columns:
        print(f"Leaking label '{label_string}' found in DataFrame columns.")
    else:
        print(f"No leaking label '{label_string}' found in DataFrame columns.")

In [ ]:
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_hc.items()}
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_hc.items()}
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_hc.items()}

{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_lbd_hc.items()}
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_lbd_hc.items()}
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_lbd_hc.items()}


{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_lbl.items()}
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_lbd.items()}
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_lbd.items()}

In [ ]:
task_hf_compensated_comb_emb_lbl_lbd_hc.get("1_1")

# OPTUNA + XGBoost

In [ ]:

#import os, json
#import numpy as np
#import pandas as pd
#import optuna, xgboost as xgb
#from typing import Dict, Tuple
#from sklearn.model_selection import StratifiedKFold, train_test_split
#from sklearn.model_selection import LeaveOneOut
#from sklearn.metrics import (
    #balanced_accuracy_score,
    #roc_auc_score,
    #confusion_matrix,
    #classification_report,
    #RocCurveDisplay,
    #ConfusionMatrixDisplay,
    #matthews_corrcoef
#)
#
## def x_y_split(df): ...  # your splitter (must return X, y)
#def _safe_feature_names(X, fallback_dim=None):
    #if hasattr(X, "columns"):
        #return list(X.columns)
    #if hasattr(X, "feature_names_in_"):
        #return list(X.feature_names_in_)
    #if fallback_dim is None and hasattr(X, "shape"):
        #fallback_dim = X.shape[1]
    #return [f"f{i}" for i in range(int(fallback_dim or 0))]
#
#def _plot_feature_importance(model, feature_names, out_dir: str):
    #"""Save multiple feature-importance views from XGBoost."""
    #import matplotlib.pyplot as plt
    #import numpy as np
    #from collections import defaultdict
    #os.makedirs(out_dir, exist_ok=True)
#
    ## 1) sklearn API importances (gain-based)
    #try:
        #importances = getattr(model, "feature_importances_", None)
        #if importances is not None and len(importances) == len(feature_names):
            #order = np.argsort(importances)[::-1]
            #top_idx = order[:50]  # limit to top-50 for readability
            #plt.figure()
            #plt.barh([feature_names[i] for i in top_idx][::-1], importances[top_idx][::-1])
            #plt.title("XGB feature_importances_ (top-50)")
            #plt.tight_layout()
            #plt.savefig(os.path.join(out_dir, "feature_importances_sklearn.png"), dpi=200, bbox_inches="tight")
            #plt.close()
    #except Exception:
        #pass
#
    ## 2) Booster importances (weight/gain/cover)
    #try:
        #booster = model.get_booster()
        #for typ in ["weight", "gain", "cover", "total_gain", "total_cover"]:
            #score_dict = booster.get_score(importance_type=typ)
            #if not score_dict:
                #continue
            ## Map 'f0'.. to friendly names
            #vals = []
            #names = []
            #for k, v in score_dict.items():
                #if k.startswith("f"):
                    #idx = int(k[1:])
                    #if 0 <= idx < len(feature_names):
                        #names.append(feature_names[idx])
                    #else:
                        #names.append(k)
                #else:
                    #names.append(k)
                #vals.append(v)
            #order = np.argsort(vals)[::-1]
            #top = min(50, len(order))
            #plt.figure()
            #plt.barh([names[i] for i in order[:top]][::-1], np.array(vals)[order[:top]][::-1])
            #plt.title(f"Booster feature importance — {typ} (top-{top})")
            #plt.tight_layout()
            #plt.savefig(os.path.join(out_dir, f"feature_importances_{typ}.png"), dpi=200, bbox_inches="tight")
            #plt.close()
    #except Exception:
        #pass
#
#def _save_shap_summaries(model, X_ref, feature_names, out_dir: str, max_display: int = 30, sample: int = 1000):
    #"""
    #Compute SHAP (TreeExplainer) and save bar + beeswarm plots.
    #Skips silently if shap not installed or backend issues arise.
    #"""
    #try:
        #import shap
        #import numpy as np
        #import matplotlib.pyplot as plt
    #except Exception:
        #return
#
    #try:
        ## Downsample for speed
        #if hasattr(X_ref, "iloc"):
            #X_use = X_ref.sample(min(sample, len(X_ref)), random_state=0)
            #X_np = X_use.values
        #else:
            #X_np = X_ref
            #if X_np.shape[0] > sample:
                #rng = np.random.default_rng(0)
                #idx = rng.choice(X_np.shape[0], size=sample, replace=False)
                #X_np = X_np[idx]
#
        #explainer = shap.TreeExplainer(model)
        #shap_values = explainer.shap_values(X_np)
#
        ## Handle binary classification: shap returns array (n, p)
        #if isinstance(shap_values, list) and len(shap_values) == 2:
            ## pick positive class
            #sv = shap_values[1]
        #else:
            #sv = shap_values
#
        ## Bar plot (mean |SHAP|)
        #try:
            #plt.figure()
            #shap.summary_plot(sv, X_np, feature_names=feature_names, plot_type="bar", show=False, max_display=max_display)
            #plt.tight_layout()
            #plt.savefig(os.path.join(out_dir, "shap_summary_bar.png"), dpi=200, bbox_inches="tight")
            #plt.close()
        #except Exception:
            #pass
#
        ## Beeswarm
        #try:
            #plt.figure()
            #shap.summary_plot(sv, X_np, feature_names=feature_names, show=False, max_display=max_display)
            #plt.tight_layout()
            #plt.savefig(os.path.join(out_dir, "shap_summary_beeswarm.png"), dpi=200, bbox_inches="tight")
            #plt.close()
        #except Exception:
            #pass
    #except Exception:
        ## Any SHAP error -> skip silently
        #return
#
#def _pick_device():
    #try:
        #_ = xgb.core.get_cuda_compute_capabilities()
        #return {"tree_method": "hist", "device": "cuda"}
    #except Exception:
        #return {"tree_method": "hist", "device": "cpu"}
#
#def _ensure_dir(path: str):
    #os.makedirs(path, exist_ok=True)
#
#def _save_json(d: dict, path: str):
    #with open(path, "w", encoding="utf-8") as f:
        #json.dump(d, f, indent=2, ensure_ascii=False)
#
#def make_xgb_objective(X, y, scoring: str = "balanced_accuracy", n_splits: int = 5, random_state: int = 42):
    #if scoring == "balanced_accuracy":
        #from sklearn.metrics import balanced_accuracy_score as metric_fn
        #needs_proba = False
        #eval_metric = "logloss"
    #elif scoring == "roc_auc":
        #from sklearn.metrics import roc_auc_score as metric_fn
        #needs_proba = True
        #eval_metric = "auc"
    #else:
        #raise ValueError("scoring must be 'balanced_accuracy' or 'roc_auc'")
#
    #folds = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    ##folds = LeaveOneOut()
    #device_kwargs = _pick_device()
#
    #pos_ratio = float(np.mean(y))
    #scale_pos_weight = ((1.0 - pos_ratio) / pos_ratio) if 0 < pos_ratio < 1 else 1.0
#
    #def objective(trial: optuna.Trial) -> float:
        #params = {
            #"max_depth": trial.suggest_int("max_depth", 3, 10),
            #"learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            #"n_estimators": trial.suggest_int("n_estimators", 200, 1500),
            #"subsample": trial.suggest_float("subsample", 0.5, 1.0),
            #"colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            #"min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            #"gamma": trial.suggest_float("gamma", 0.0, 5.0),
            #"reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            #"reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            #"max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            #"objective": "binary:logistic",
            #"eval_metric": eval_metric,
            #"n_jobs": 1,  # keep each trial single-threaded
            #"scale_pos_weight": scale_pos_weight,
            #**device_kwargs,
        #}
#
        #scores = []
        #for tr_idx, va_idx in folds.split(X, y):
            #if hasattr(X, "iloc"):
                #X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
                #y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
            #else:
                #X_tr, X_va = X[tr_idx], X[va_idx]
                #y_tr, y_va = y[tr_idx], y[va_idx]
#
            #model = xgb.XGBClassifier(**params)
            #model.fit(
                #X_tr, y_tr,
                #eval_set=[(X_va, y_va)],
                ##early_stopping_rounds=50,
                #verbose=False,
            #)
#
            #if needs_proba:
                #y_score = model.predict_proba(X_va)[:, 1]
                #score = metric_fn(y_va, y_score)
            #else:
                #y_pred = model.predict(X_va)
                #score = metric_fn(y_va, y_pred)
            #scores.append(score)
#
            #trial.report(np.mean(scores), len(scores))
            #if trial.should_prune():
                #raise optuna.TrialPruned()
#
        #return float(np.mean(scores))
#
    #return objective
#
#def _evaluate_and_save(name: str, model, X_test, y_test, out_dir: str, feature_names=None, X_ref_for_shap=None):
    #"""Compute metrics, plot ROC + confusion matrix, and save everything."""
    #_ensure_dir(out_dir)
#
    ## Predictions
    #y_proba = None
    #try:
        #y_proba = model.predict_proba(X_test)[:, 1]
    #except Exception:
        #pass
    #y_pred = model.predict(X_test)
#
    ## Metrics
    #metrics = {
        #"balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        #"mcc": None,
        #"roc_auc": None,
        #"sensitivity": None,   # <-- NEW
        #"specificity": None,   # <-- NEW
        #"classification_report": None,
    #}
    ## MCC
    #try:
        #metrics["mcc"] = float(matthews_corrcoef(y_test, y_pred))
    #except Exception:
        #pass
    ## ROC-AUC if both classes present and proba available
    #try:
        #if y_proba is not None and len(np.unique(y_test)) == 2:
            #metrics["roc_auc"] = float(roc_auc_score(y_test, y_proba))
        #else:
            #metrics["roc_auc"] = None
    #except Exception:
        #metrics["roc_auc"] = None
    ## --- Sensitivity & Specificity (labels fixed to [0,1]) ---
    #try:
        #cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        #tn, fp, fn, tp = cm.ravel()
        #sens_den = tp + fn
        #spec_den = tn + fp
        #metrics["sensitivity"] = float(tp / sens_den) if sens_den > 0 else None
        #metrics["specificity"] = float(tn / spec_den) if spec_den > 0 else None
    #except Exception:
        #pass
#
#
    ## Classification report (as dict)
    #try:
        #metrics["classification_report"] = classification_report(y_test, y_pred, output_dict=True)
    #except Exception:
        #metrics["classification_report"] = None
#
    ## Confusion matrix plot
    #try:
        #cm = confusion_matrix(y_test, y_pred)
        #disp = ConfusionMatrixDisplay(cm)
        #import matplotlib.pyplot as plt
        #plt.figure()
        #disp.plot(values_format="d")
        #plt.title(f"Confusion Matrix — {name}")
        #plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=200, bbox_inches="tight")
        #plt.close()
    #except Exception:
        #pass
#
    ## ROC curve plot
    #try:
        #if y_proba is not None and len(np.unique(y_test)) == 2:
            #import matplotlib.pyplot as plt
            #plt.figure()
            #RocCurveDisplay.from_predictions(y_test, y_proba)
            #plt.title(f"ROC Curve — {name}")
            #plt.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=200, bbox_inches="tight")
            #plt.close()
    #except Exception:
        #pass
#
    ## Feature names
    #if feature_names is None:
        #feature_names = _safe_feature_names(X_test)
#
    ## Feature importance plots
    #try:
        #_plot_feature_importance(model, feature_names, out_dir)
    #except Exception:
        #pass
#
    ## SHAP plots (optional)
    #try:
        #if X_ref_for_shap is None:
            #X_ref_for_shap = X_test
        #_save_shap_summaries(model, X_ref_for_shap, feature_names, out_dir)
    #except Exception:
        #pass
#
    ## Save metrics JSON
    #_save_json(metrics, os.path.join(out_dir, "metrics.json"))
    #return metrics
#
#def optimize_many(
    #dfs: Dict[str, pd.DataFrame],
    #x_y_split_fn,
    #scoring: str = "balanced_accuracy",
    #n_trials: int = 500,
    #n_splits: int = 5,
    #random_state: int = 42,
    #n_jobs_trials: int = -1,
    #output_dir: str = "xgb_results",
    #test_size: float = 0.20,
    #split_param: str = "emb_",
#) -> Tuple[pd.DataFrame, Dict[str, xgb.XGBClassifier], Dict[str, optuna.Study]]:
    #"""
    #Runs Optuna per dataset, evaluates on a held-out test set,
    #and saves plots + metrics in output_dir/<dataset_name>/
    #"""
    #results = []
    #best_models: Dict[str, xgb.XGBClassifier] = {}
    #studies: Dict[str, optuna.Study] = {}
#
    #_ensure_dir(output_dir)
#
    #for name, df in dfs.items():
        #X, y = x_y_split_fn(df, split_param=split_param)
#
        ## To numpy for fast indexing, but keep DataFrame support
        #X_arr = X.values if hasattr(X, "values") else X
        #y_arr = y.values if hasattr(y, "values") else y
#
        #feature_names = _safe_feature_names(X)
        ## Keep names before converting to numpy
#
        ## Hold-out test split (kept untouched for final evaluation)
        #X_train, X_test, y_train, y_test = train_test_split(
            #X_arr, y_arr, test_size=test_size, random_state=random_state, stratify=y_arr
        #)
#
        ## Build objective on TRAIN ONLY
        #objective = make_xgb_objective(X_train, y_train, scoring=scoring, n_splits=n_splits, random_state=random_state)
#
        #study = optuna.create_study(
            #study_name=f"{name}_{scoring}",
            #direction="maximize",
            #sampler=optuna.samplers.TPESampler(seed=random_state),
            #pruner=optuna.pruners.MedianPruner(n_startup_trials=10),
        #)
        #study.optimize(objective, n_trials=n_trials, show_progress_bar=True, n_jobs=n_jobs_trials)
        #studies[name] = study
#
        ## Train final model on TRAIN using best params (with small VA split for early stopping)
        #device_kwargs = _pick_device()
        #pos_ratio = float(np.mean(y_train))
        #spw = ((1.0 - pos_ratio) / pos_ratio) if 0 < pos_ratio < 1 else 1.0
#
        #best_params = {
            #**study.best_params,
            #"objective": "binary:logistic",
            #"eval_metric": "auc" if scoring == "roc_auc" else "logloss",
            #"n_jobs": 0,
            #"scale_pos_weight": spw,
            #**device_kwargs,
        #}
        #final_model = xgb.XGBClassifier(**best_params)
#
        ## Early stopping split inside TRAIN
        #X_tr, X_va, y_tr, y_va = train_test_split(
            #X_train, y_train, test_size=0.15, random_state=random_state, stratify=y_train
        #)
        #final_model.fit(
            #X_tr, y_tr,
            #eval_set=[(X_va, y_va)],
            ##early_stopping_rounds=50,
            #verbose=False,
        #)
#
        ## Save per-dataset artifacts
        #ds_out = os.path.join(output_dir, name)
        #_ensure_dir(ds_out)
#
        ## Evaluate on TEST and save plots/metrics
        #metrics = _evaluate_and_save(
            #name, final_model, X_test, y_test, ds_out,
            #feature_names=feature_names,
            #X_ref_for_shap=X_test
        #)
#
        ## Save best params + study summary
        #_save_json({"best_params": study.best_params, "best_value": study.best_value}, os.path.join(ds_out, "best.json"))
#
        ## Optional: save model
        #try:
            #import joblib
            #joblib.dump(final_model, os.path.join(ds_out, "model.joblib"))
        #except Exception:
            #pass
#
        #best_models[name] = final_model
        #results.append({
            #"dataset": name,
            #"cv_best_value": study.best_value,
            #"test_balanced_accuracy": metrics.get("balanced_accuracy"),
            #"test_roc_auc": metrics.get("roc_auc"),
            #"n_trials": len(study.trials),
        #})
#
    #summary_df = pd.DataFrame(results).sort_values(by="test_balanced_accuracy", ascending=False).reset_index(drop=True)
    ## Save a global summary too
    #summary_df.to_csv(os.path.join(output_dir, "summary.csv"), index=False)
    #return summary_df, best_models, studies
#

In [ ]:
#import os, json
#import numpy as np
#import pandas as pd
#import optuna, xgboost as xgb
#from typing import Dict, Tuple
#from sklearn.model_selection import StratifiedKFold, train_test_split, LeaveOneOut
#from sklearn.metrics import (
    #balanced_accuracy_score,
    #roc_auc_score,
    #confusion_matrix,
    #classification_report,
    #RocCurveDisplay,
    #ConfusionMatrixDisplay,
    #matthews_corrcoef
#)
#
## def x_y_split(df): ...  # your splitter (must return X, y)
#def _safe_feature_names(X, fallback_dim=None):
    #if hasattr(X, "columns"):
        #return list(X.columns)
    #if hasattr(X, "feature_names_in_"):
        #return list(X.feature_names_in_)
    #if fallback_dim is None and hasattr(X, "shape"):
        #fallback_dim = X.shape[1]
    #return [f"f{i}" for i in range(int(fallback_dim or 0))]
#
#def _plot_feature_importance(model, feature_names, out_dir: str):
    #"""Save multiple feature-importance views from XGBoost."""
    #import matplotlib.pyplot as plt
    #import numpy as np
    #from collections import defaultdict
    #os.makedirs(out_dir, exist_ok=True)
#
    ## 1) sklearn API importances (gain-based)
    #try:
        #importances = getattr(model, "feature_importances_", None)
        #if importances is not None and len(importances) == len(feature_names):
            #order = np.argsort(importances)[::-1]
            #top_idx = order[:50]  # limit to top-50 for readability
            #plt.figure()
            #plt.barh([feature_names[i] for i in top_idx][::-1], importances[top_idx][::-1])
            #plt.title("XGB feature_importances_ (top-50)")
            #plt.tight_layout()
            #plt.savefig(os.path.join(out_dir, "feature_importances_sklearn.png"), dpi=200, bbox_inches="tight")
            #plt.close()
    #except Exception:
        #pass
#
    ## 2) Booster importances (weight/gain/cover)
    #try:
        #booster = model.get_booster()
        #for typ in ["weight", "gain", "cover", "total_gain", "total_cover"]:
            #score_dict = booster.get_score(importance_type=typ)
            #if not score_dict:
                #continue
            ## Map 'f0'.. to friendly names
            #vals = []
            #names = []
            #for k, v in score_dict.items():
                #if k.startswith("f"):
                    #idx = int(k[1:])
                    #if 0 <= idx < len(feature_names):
                        #names.append(feature_names[idx])
                    #else:
                        #names.append(k)
                #else:
                    #names.append(k)
                #vals.append(v)
            #order = np.argsort(vals)[::-1]
            #top = min(50, len(order))
            #plt.figure()
            #plt.barh([names[i] for i in order[:top]][::-1], np.array(vals)[order[:top]][::-1])
            #plt.title(f"Booster feature importance — {typ} (top-{top})")
            #plt.tight_layout()
            #plt.savefig(os.path.join(out_dir, f"feature_importances_{typ}.png"), dpi=200, bbox_inches="tight")
            #plt.close()
    #except Exception:
        #pass
#
#def _save_shap_summaries(model, X_ref, feature_names, out_dir: str, max_display: int = 30, sample: int = 1000):
    #"""
    #Compute SHAP (TreeExplainer) and save bar + beeswarm plots.
    #Skips silently if shap not installed or backend issues arise.
    #"""
    #try:
        #import shap
        #import numpy as np
        #import matplotlib.pyplot as plt
    #except Exception:
        #return
#
    #try:
        ## Downsample for speed
        #if hasattr(X_ref, "iloc"):
            #X_use = X_ref.sample(min(sample, len(X_ref)), random_state=0)
            #X_np = X_use.values
        #else:
            #X_np = X_ref
            #if X_np.shape[0] > sample:
                #rng = np.random.default_rng(0)
                #idx = rng.choice(X_np.shape[0], size=sample, replace=False)
                #X_np = X_np[idx]
#
        #explainer = shap.TreeExplainer(model)
        #shap_values = explainer.shap_values(X_np)
#
        ## Handle binary classification: shap returns array (n, p)
        #if isinstance(shap_values, list) and len(shap_values) == 2:
            ## pick positive class
            #sv = shap_values[1]
        #else:
            #sv = shap_values
#
        ## Bar plot (mean |SHAP|)
        #try:
            #plt.figure()
            #shap.summary_plot(sv, X_np, feature_names=feature_names, plot_type="bar", show=False, max_display=max_display)
            #plt.tight_layout()
            #plt.savefig(os.path.join(out_dir, "shap_summary_bar.png"), dpi=200, bbox_inches="tight")
            #plt.close()
        #except Exception:
            #pass
#
        ## Beeswarm
        #try:
            #plt.figure()
            #shap.summary_plot(sv, X_np, feature_names=feature_names, show=False, max_display=max_display)
            #plt.tight_layout()
            #plt.savefig(os.path.join(out_dir, "shap_summary_beeswarm.png"), dpi=200, bbox_inches="tight")
            #plt.close()
        #except Exception:
            #pass
    #except Exception:
        ## Any SHAP error -> skip silently
        #return
#
#def _pick_device():
    #try:
        #_ = xgb.core.get_cuda_compute_capabilities()
        #return {"tree_method": "hist", "device": "cuda"}
    #except Exception:
        #return {"tree_method": "hist", "device": "cpu"}
#
#def _ensure_dir(path: str):
    #os.makedirs(path, exist_ok=True)
#
#def _save_json(d: dict, path: str):
    #with open(path, "w", encoding="utf-8") as f:
        #json.dump(d, f, indent=2, ensure_ascii=False)
#
#def make_xgb_objective(X, y, scoring: str = "balanced_accuracy", cv_strategy: str = "loo", 
                       #n_splits: int = 5, random_state: int = 42):
    #"""
    #Create Optuna objective function with configurable CV strategy.
    #
    #Parameters:
    #-----------
    #cv_strategy : str
        #Either 'skf' (StratifiedKFold) or 'loo' (LeaveOneOut)
    #n_splits : int
        #Number of splits for StratifiedKFold (ignored for LOO)
    #"""
    #if scoring == "balanced_accuracy":
        #from sklearn.metrics import balanced_accuracy_score as metric_fn
        #needs_proba = False
        #eval_metric = "logloss"
    #elif scoring == "roc_auc":
        #from sklearn.metrics import roc_auc_score as metric_fn
        #needs_proba = True
        #eval_metric = "auc"
    #else:
        #raise ValueError("scoring must be 'balanced_accuracy' or 'roc_auc'")
#
    ## Choose CV strategy
    #if cv_strategy == "loo":
        #folds = LeaveOneOut()
        #print(f"Using Leave-One-Out CV with {len(y)} folds")
    #elif cv_strategy == "skf":
        #folds = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        #print(f"Using Stratified K-Fold CV with {n_splits} folds")
    #else:
        #raise ValueError("cv_strategy must be 'skf' or 'loo'")
#
    #device_kwargs = _pick_device()
#
    #pos_ratio = float(np.mean(y))
    #scale_pos_weight = ((1.0 - pos_ratio) / pos_ratio) if 0 < pos_ratio < 1 else 1.0
#
    #def objective(trial: optuna.Trial) -> float:
        #params = {
            #"max_depth": trial.suggest_int("max_depth", 3, 10),
            #"learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            #"n_estimators": trial.suggest_int("n_estimators", 200, 1500),
            #"subsample": trial.suggest_float("subsample", 0.5, 1.0),
            #"colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            #"min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            #"gamma": trial.suggest_float("gamma", 0.0, 5.0),
            #"reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            #"reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            #"max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            #"objective": "binary:logistic",
            #"eval_metric": eval_metric,
            #"n_jobs": 1,  # keep each trial single-threaded
            #"scale_pos_weight": scale_pos_weight,
            #**device_kwargs,
        #}
#
        #scores = []
        #for fold_idx, (tr_idx, va_idx) in enumerate(folds.split(X, y)):
            #if hasattr(X, "iloc"):
                #X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
                #y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
            #else:
                #X_tr, X_va = X[tr_idx], X[va_idx]
                #y_tr, y_va = y[tr_idx], y[va_idx]
#
            #model = xgb.XGBClassifier(**params)
            #
            ## For LOO, skip early stopping (only 1 validation sample)
            #if cv_strategy == "loo":
                #model.fit(X_tr, y_tr, verbose=False)
            #else:
                #model.fit(
                    #X_tr, y_tr,
                    #eval_set=[(X_va, y_va)],
                    #verbose=False,
                #)
#
            #if needs_proba:
                #y_score = model.predict_proba(X_va)[:, 1]
                #score = metric_fn(y_va, y_score)
            #else:
                #y_pred = model.predict(X_va)
                #score = metric_fn(y_va, y_pred)
            #scores.append(score)
#
            ## Report intermediate values and check for pruning
            ## For LOO: report every 10 folds to reduce overhead
            #if cv_strategy == "loo":
                #if (fold_idx + 1) % 10 == 0:
                    #trial.report(np.mean(scores), fold_idx)
                    #if trial.should_prune():
                        #raise optuna.TrialPruned()
            #else:
                #trial.report(np.mean(scores), fold_idx)
                #if trial.should_prune():
                    #raise optuna.TrialPruned()
#
        #return float(np.mean(scores))
#
    #return objective
#
#def _evaluate_and_save(name: str, model, X_test, y_test, out_dir: str, feature_names=None, X_ref_for_shap=None):
    #"""Compute metrics, plot ROC + confusion matrix, and save everything."""
    #_ensure_dir(out_dir)
#
    ## Predictions
    #y_proba = None
    #try:
        #y_proba = model.predict_proba(X_test)[:, 1]
    #except Exception:
        #pass
    #y_pred = model.predict(X_test)
#
    ## Metrics
    #metrics = {
        #"balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        #"mcc": None,
        #"roc_auc": None,
        #"sensitivity": None,
        #"specificity": None,
        #"classification_report": None,
    #}
    ## MCC
    #try:
        #metrics["mcc"] = float(matthews_corrcoef(y_test, y_pred))
    #except Exception:
        #pass
    ## ROC-AUC if both classes present and proba available
    #try:
        #if y_proba is not None and len(np.unique(y_test)) == 2:
            #metrics["roc_auc"] = float(roc_auc_score(y_test, y_proba))
        #else:
            #metrics["roc_auc"] = None
    #except Exception:
        #metrics["roc_auc"] = None
    ## --- Sensitivity & Specificity (labels fixed to [0,1]) ---
    #try:
        #cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        #tn, fp, fn, tp = cm.ravel()
        #sens_den = tp + fn
        #spec_den = tn + fp
        #metrics["sensitivity"] = float(tp / sens_den) if sens_den > 0 else None
        #metrics["specificity"] = float(tn / spec_den) if spec_den > 0 else None
    #except Exception:
        #pass
#
    ## Classification report (as dict)
    #try:
        #metrics["classification_report"] = classification_report(y_test, y_pred, output_dict=True)
    #except Exception:
        #metrics["classification_report"] = None
#
    ## Confusion matrix plot
    #try:
        #cm = confusion_matrix(y_test, y_pred)
        #disp = ConfusionMatrixDisplay(cm)
        #import matplotlib.pyplot as plt
        #plt.figure()
        #disp.plot(values_format="d")
        #plt.title(f"Confusion Matrix — {name}")
        #plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=200, bbox_inches="tight")
        #plt.close()
    #except Exception:
        #pass
#
    ## ROC curve plot
    #try:
        #if y_proba is not None and len(np.unique(y_test)) == 2:
            #import matplotlib.pyplot as plt
            #plt.figure()
            #RocCurveDisplay.from_predictions(y_test, y_proba)
            #plt.title(f"ROC Curve — {name}")
            #plt.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=200, bbox_inches="tight")
            #plt.close()
    #except Exception:
        #pass
#
    ## Feature names
    #if feature_names is None:
        #feature_names = _safe_feature_names(X_test)
#
    ## Feature importance plots
    #try:
        #_plot_feature_importance(model, feature_names, out_dir)
    #except Exception:
        #pass
#
    ## SHAP plots (optional)
    #try:
        #if X_ref_for_shap is None:
            #X_ref_for_shap = X_test
        #_save_shap_summaries(model, X_ref_for_shap, feature_names, out_dir)
    #except Exception:
        #pass
#
    ## Save metrics JSON
    #_save_json(metrics, os.path.join(out_dir, "metrics.json"))
    #return metrics
#
#def optimize_many(
    #dfs: Dict[str, pd.DataFrame],
    #x_y_split_fn,
    #scoring: str = "balanced_accuracy",
    #cv_strategy: str = "loo",  # NEW: 'skf' or 'loo'
    #n_trials: int = 500,
    #n_splits: int = 5,
    #random_state: int = 42,
    #n_jobs_trials: int = -1,
    #output_dir: str = "xgb_results",
    #test_size: float = 0.20,
    #split_param: str = "emb_",
#) -> Tuple[pd.DataFrame, Dict[str, xgb.XGBClassifier], Dict[str, optuna.Study]]:
    #"""
    #Runs Optuna per dataset, evaluates on a held-out test set,
    #and saves plots + metrics in output_dir/<dataset_name>/
    #
    #Parameters:
    #-----------
    #cv_strategy : str
        #Either 'skf' (StratifiedKFold) or 'loo' (LeaveOneOut)
    #"""
    #results = []
    #best_models: Dict[str, xgb.XGBClassifier] = {}
    #studies: Dict[str, optuna.Study] = {}
#
    #_ensure_dir(output_dir)
#
    #for name, df in dfs.items():
        #print(f"\n{'='*60}")
        #print(f"Processing dataset: {name}")
        #print(f"{'='*60}")
        #
        #X, y = x_y_split_fn(df, split_param=split_param)
#
        ## To numpy for fast indexing, but keep DataFrame support
        #X_arr = X.values if hasattr(X, "values") else X
        #y_arr = y.values if hasattr(y, "values") else y
#
        #feature_names = _safe_feature_names(X)
#
        ## Hold-out test split (kept untouched for final evaluation)
        #X_train, X_test, y_train, y_test = train_test_split(
            #X_arr, y_arr, test_size=test_size, random_state=random_state, stratify=y_arr
        #)
#
        #print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
#
        ## Build objective on TRAIN ONLY
        #objective = make_xgb_objective(
            #X_train, y_train, 
            #scoring=scoring, 
            #cv_strategy=cv_strategy,
            #n_splits=n_splits, 
            #random_state=random_state
        #)
#
        #study = optuna.create_study(
            #study_name=f"{name}_{scoring}",
            #direction="maximize",
            #sampler=optuna.samplers.TPESampler(seed=random_state),
            #pruner=optuna.pruners.MedianPruner(n_startup_trials=10),
        #)
        #
        #print(f"Starting optimization with {n_trials} trials...")
        #study.optimize(objective, n_trials=n_trials, show_progress_bar=True, n_jobs=n_jobs_trials)
        #studies[name] = study
#
        ## Train final model on TRAIN using best params
        #device_kwargs = _pick_device()
        #pos_ratio = float(np.mean(y_train))
        #spw = ((1.0 - pos_ratio) / pos_ratio) if 0 < pos_ratio < 1 else 1.0
#
        #best_params = {
            #**study.best_params,
            #"objective": "binary:logistic",
            #"eval_metric": "auc" if scoring == "roc_auc" else "logloss",
            #"n_jobs": 0,
            #"scale_pos_weight": spw,
            #**device_kwargs,
        #}
        #final_model = xgb.XGBClassifier(**best_params)
#
        ## For final model training:
        ## - If LOO was used in CV, train on full training set
        ## - If SKF was used, use early stopping split
        #if cv_strategy == "loo":
            #print("Training final model on full training set (LOO strategy)...")
            #final_model.fit(X_train, y_train, verbose=False)
        #else:
            #print("Training final model with early stopping validation split...")
            #X_tr, X_va, y_tr, y_va = train_test_split(
                #X_train, y_train, test_size=0.15, random_state=random_state, stratify=y_train
            #)
            #final_model.fit(
                #X_tr, y_tr,
                #eval_set=[(X_va, y_va)],
                #verbose=False,
            #)
#
        ## Save per-dataset artifacts
        #ds_out = os.path.join(output_dir, name)
        #_ensure_dir(ds_out)
#
        ## Evaluate on TEST and save plots/metrics
        #print("Evaluating on test set...")
        #metrics = _evaluate_and_save(
            #name, final_model, X_test, y_test, ds_out,
            #feature_names=feature_names,
            #X_ref_for_shap=X_test
        #)
#
        ## Save best params + study summary
        #_save_json({
            #"best_params": study.best_params, 
            #"best_value": study.best_value,
            #"cv_strategy": cv_strategy,
            #"n_folds": len(y_train) if cv_strategy == "loo" else n_splits
        #}, os.path.join(ds_out, "best.json"))
#
        ## Optional: save model
        #try:
            #import joblib
            #joblib.dump(final_model, os.path.join(ds_out, "model.joblib"))
        #except Exception:
            #pass
#
        #best_models[name] = final_model
        #results.append({
            #"dataset": name,
            #"cv_strategy": cv_strategy,
            #"cv_best_value": study.best_value,
            #"test_balanced_accuracy": metrics.get("balanced_accuracy"),
            #"test_roc_auc": metrics.get("roc_auc"),
            #"test_mcc": metrics.get("mcc"),
            #"test_sensitivity": metrics.get("sensitivity"),
            #"test_specificity": metrics.get("specificity"),
            #"n_trials": len(study.trials),
            #"train_size": len(X_train),
            #"test_size": len(X_test),
        #})
#
    #summary_df = pd.DataFrame(results).sort_values(by="test_balanced_accuracy", ascending=False).reset_index(drop=True)
    ## Save a global summary too
    #summary_df.to_csv(os.path.join(output_dir, "summary.csv"), index=False)
    #
    #print(f"\n{'='*60}")
    #print("All datasets processed. Summary:")
    #print(f"{'='*60}")
    #print(summary_df)
    #
    #return summary_df, best_models, studies

In [ ]:
import os, json
import numpy as np
import pandas as pd
import optuna, xgboost as xgb
from typing import Dict, Tuple
from sklearn.model_selection import StratifiedKFold, train_test_split, LeaveOneOut
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
    matthews_corrcoef
)

# def x_y_split(df): ...  # your splitter (must return X, y)
def _safe_feature_names(X, fallback_dim=None):
    if hasattr(X, "columns"):
        return list(X.columns)
    if hasattr(X, "feature_names_in_"):
        return list(X.feature_names_in_)
    if fallback_dim is None and hasattr(X, "shape"):
        fallback_dim = X.shape[1]
    return [f"f{i}" for i in range(int(fallback_dim or 0))]

def _plot_feature_importance(model, feature_names, out_dir: str):
    """Save multiple feature-importance views from XGBoost."""
    import matplotlib.pyplot as plt
    import numpy as np
    from collections import defaultdict
    os.makedirs(out_dir, exist_ok=True)

    # 1) sklearn API importances (gain-based)
    try:
        importances = getattr(model, "feature_importances_", None)
        if importances is not None and len(importances) == len(feature_names):
            order = np.argsort(importances)[::-1]
            top_idx = order[:50]  # limit to top-50 for readability
            plt.figure()
            plt.barh([feature_names[i] for i in top_idx][::-1], importances[top_idx][::-1])
            plt.title("XGB feature_importances_ (top-50)")
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "feature_importances_sklearn.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

    # 2) Booster importances (weight/gain/cover)
    try:
        booster = model.get_booster()
        for typ in ["weight", "gain", "cover", "total_gain", "total_cover"]:
            score_dict = booster.get_score(importance_type=typ)
            if not score_dict:
                continue
            vals, names = [], []
            for k, v in score_dict.items():
                if k.startswith("f"):
                    idx = int(k[1:])
                    names.append(feature_names[idx] if 0 <= idx < len(feature_names) else k)
                else:
                    names.append(k)
                vals.append(v)
            order = np.argsort(vals)[::-1]
            top = min(50, len(order))
            plt.figure()
            plt.barh([names[i] for i in order[:top]][::-1], np.array(vals)[order[:top]][::-1])
            plt.title(f"Booster feature importance — {typ} (top-{top})")
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"feature_importances_{typ}.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

def _save_shap_summaries(model, X_ref, feature_names, out_dir: str, max_display: int = 30, sample: int = 1000):
    """
    Compute SHAP (TreeExplainer) and save bar + beeswarm plots.
    Skips silently if shap not installed or backend issues arise.
    """
    try:
        import shap
        import numpy as np
        import matplotlib.pyplot as plt
    except Exception:
        return

    try:
        if hasattr(X_ref, "iloc"):
            X_use = X_ref.sample(min(sample, len(X_ref)), random_state=0)
            X_np = X_use.values
        else:
            X_np = X_ref
            if X_np.shape[0] > sample:
                rng = np.random.default_rng(0)
                idx = rng.choice(X_np.shape[0], size=sample, replace=False)
                X_np = X_np[idx]

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_np)
        sv = shap_values[1] if isinstance(shap_values, list) and len(shap_values) == 2 else shap_values

        try:
            plt.figure()
            shap.summary_plot(sv, X_np, feature_names=feature_names, plot_type="bar", show=False, max_display=max_display)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "shap_summary_bar.png"), dpi=200, bbox_inches="tight")
            plt.close()
        except Exception:
            pass

        try:
            plt.figure()
            shap.summary_plot(sv, X_np, feature_names=feature_names, show=False, max_display=max_display)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "shap_summary_beeswarm.png"), dpi=200, bbox_inches="tight")
            plt.close()
        except Exception:
            pass
    except Exception:
        return

def _pick_device():
    try:
        _ = xgb.core.get_cuda_compute_capabilities()
        return {"tree_method": "hist", "device": "cuda"}
    except Exception:
        return {"tree_method": "hist", "device": "cpu"}

def _ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def _save_json(d: dict, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(d, f, indent=2, ensure_ascii=False)

def make_xgb_objective(
    X, y,
    scoring: str = "balanced_accuracy",
    cv_strategy: str = "loo",
    n_splits: int = 5,
    random_state: int = 42
):
    """Create Optuna objective with configurable CV strategy."""
    if scoring == "balanced_accuracy":
        from sklearn.metrics import balanced_accuracy_score as metric_fn
        needs_proba = False
        eval_metric = "logloss"
    elif scoring == "roc_auc":
        from sklearn.metrics import roc_auc_score as metric_fn
        needs_proba = True
        eval_metric = "auc"
    else:
        raise ValueError("scoring must be 'balanced_accuracy' or 'roc_auc'")

    if cv_strategy == "loo":
        folds = LeaveOneOut()
        print(f"Using Leave-One-Out CV with {len(y)} folds")
    elif cv_strategy == "skf":
        folds = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        print(f"Using Stratified K-Fold CV with {n_splits} folds")
    else:
        raise ValueError("cv_strategy must be 'skf' or 'loo'")

    device_kwargs = _pick_device()
    pos_ratio = float(np.mean(y))
    scale_pos_weight = ((1.0 - pos_ratio) / pos_ratio) if 0 < pos_ratio < 1 else 1.0

    y_array = y.to_numpy() if hasattr(y, "to_numpy") else np.asarray(y)
    n_samples = len(y_array)

    def objective(trial: optuna.Trial) -> float:
        params = {
            #To Tune leabes, branches, trees, regularization
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 200, 1500),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "objective": "binary:logistic",
            "eval_metric": eval_metric,
            "n_jobs": 1,
            "scale_pos_weight": scale_pos_weight,
            **device_kwargs,
        }

        oof_predictions = np.zeros(n_samples, dtype=float)
        filled_mask = np.zeros(n_samples, dtype=bool)

        for fold_idx, (tr_idx, va_idx) in enumerate(folds.split(X, y)):
            if hasattr(X, "iloc"):
                X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
                y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
            else:
                X_tr, X_va = X[tr_idx], X[va_idx]
                y_tr, y_va = y[tr_idx], y[va_idx]

            model = xgb.XGBClassifier(**params)
            model.fit(X_tr, y_tr, verbose=False)

            preds = model.predict_proba(X_va)[:, 1] if needs_proba else model.predict(X_va)
            oof_predictions[va_idx] = preds
            filled_mask[va_idx] = True

            y_seen = y_array[filled_mask]
            preds_seen = oof_predictions[filled_mask]

            if np.unique(y_seen).size < 2:
                continue

            score = metric_fn(y_seen, preds_seen)
            if cv_strategy == "loo":
                if (fold_idx + 1) % 10 == 0:
                    trial.report(score, fold_idx)
                    if trial.should_prune():
                        raise optuna.TrialPruned()
            else:
                trial.report(score, fold_idx)
                if trial.should_prune():
                    raise optuna.TrialPruned()

        final_score = metric_fn(y_array, oof_predictions)
        return float(final_score)

    return objective

def _evaluate_and_save(name: str, model, X_test, y_test, out_dir: str, feature_names=None, X_ref_for_shap=None):
    """Compute metrics, plot ROC + confusion matrix, and save everything."""
    _ensure_dir(out_dir)

    y_proba = None
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
    except Exception:
        pass
    y_pred = model.predict(X_test)

    metrics = {
        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        "mcc": None,
        "roc_auc": None,
        "sensitivity": None,
        "specificity": None,
        "classification_report": None,
    }
    try:
        metrics["mcc"] = float(matthews_corrcoef(y_test, y_pred))
    except Exception:
        pass
    try:
        if y_proba is not None and len(np.unique(y_test)) == 2:
            metrics["roc_auc"] = float(roc_auc_score(y_test, y_proba))
        else:
            metrics["roc_auc"] = None
    except Exception:
        metrics["roc_auc"] = None
    try:
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        sens_den = tp + fn
        spec_den = tn + fp
        metrics["sensitivity"] = float(tp / sens_den) if sens_den > 0 else None
        metrics["specificity"] = float(tn / spec_den) if spec_den > 0 else None
    except Exception:
        pass
    try:
        metrics["classification_report"] = classification_report(y_test, y_pred, output_dict=True)
    except Exception:
        metrics["classification_report"] = None

    try:
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(cm)
        import matplotlib.pyplot as plt
        plt.figure()
        disp.plot(values_format="d")
        plt.title(f"Confusion Matrix — {name}")
        plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=200, bbox_inches="tight")
        plt.close()
    except Exception:
        pass

    try:
        if y_proba is not None and len(np.unique(y_test)) == 2:
            import matplotlib.pyplot as plt
            plt.figure()
            RocCurveDisplay.from_predictions(y_test, y_proba)
            plt.title(f"ROC Curve — {name}")
            plt.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

    if feature_names is None:
        feature_names = _safe_feature_names(X_test)

    try:
        _plot_feature_importance(model, feature_names, out_dir)
    except Exception:
        pass

    try:
        if X_ref_for_shap is None:
            X_ref_for_shap = X_test
        _save_shap_summaries(model, X_ref_for_shap, feature_names, out_dir)
    except Exception:
        pass

    _save_json(metrics, os.path.join(out_dir, "metrics.json"))
    return metrics

def optimize_many(
    dfs: Dict[str, pd.DataFrame],
    x_y_split_fn,
    scoring: str = "balanced_accuracy",
    cv_strategy: str = "loo",
    n_trials: int = 500,
    n_splits: int = 5,
    random_state: int = 42,
    n_jobs_trials: int = -1,
    output_dir: str = "xgb_results",
    test_size: float = 0.20,
    split_param: str = "emb_",
) -> Tuple[pd.DataFrame, Dict[str, xgb.XGBClassifier], Dict[str, optuna.Study]]:
    """
    Runs Optuna per dataset, evaluates on a held-out test set,
    and saves plots + metrics in output_dir/<dataset_name>/
    """
    results = []
    best_models: Dict[str, xgb.XGBClassifier] = {}
    studies: Dict[str, optuna.Study] = {}

    _ensure_dir(output_dir)

    for name, df in dfs.items():
        print(f"\n{'='*60}")
        print(f"Processing dataset: {name}")
        print(f"{'='*60}")
        
        X, y = x_y_split_fn(df, split_param=split_param)

        X_arr = X.values if hasattr(X, "values") else X
        y_arr = y.values if hasattr(y, "values") else y

        feature_names = _safe_feature_names(X)

        X_train, X_test, y_train, y_test = train_test_split(
            X_arr, y_arr, test_size=test_size, random_state=random_state, stratify=y_arr
        )

        print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

        objective = make_xgb_objective(
            X_train, y_train,
            scoring=scoring,
            cv_strategy=cv_strategy,
            n_splits=n_splits,
            random_state=random_state
        )

        study = optuna.create_study(
            study_name=f"{name}_{scoring}",
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=random_state),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=10),
        )
        
        print(f"Starting optimization with {n_trials} trials...")
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True, n_jobs=n_jobs_trials)
        studies[name] = study

        device_kwargs = _pick_device()
        pos_ratio = float(np.mean(y_train))
        spw = ((1.0 - pos_ratio) / pos_ratio) if 0 < pos_ratio < 1 else 1.0

        best_params = {
            **study.best_params,
            "objective": "binary:logistic",
            "eval_metric": "auc" if scoring == "roc_auc" else "logloss",
            "n_jobs": 0,
            "scale_pos_weight": spw,
            **device_kwargs,
        }
        final_model = xgb.XGBClassifier(**best_params)

        if cv_strategy == "loo":
            print("Training final model on full training set (LOO strategy)...")
            final_model.fit(X_train, y_train, verbose=False)
        else:
            print("Training final model with early stopping validation split...")
            X_tr, X_va, y_tr, y_va = train_test_split(
                X_train, y_train, test_size=0.15, random_state=random_state, stratify=y_train
            )
            final_model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                verbose=False,
            )

        ds_out = os.path.join(output_dir, name)
        _ensure_dir(ds_out)

        print("Evaluating on test set...")
        metrics = _evaluate_and_save(
            name, final_model, X_test, y_test, ds_out,
            feature_names=feature_names,
            X_ref_for_shap=X_test
        )

        _save_json({
            "best_params": study.best_params,
            "best_value": study.best_value,
            "cv_strategy": cv_strategy,
            "n_folds": len(y_train) if cv_strategy == "loo" else n_splits
        }, os.path.join(ds_out, "best.json"))

        try:
            import joblib
            joblib.dump(final_model, os.path.join(ds_out, "model.joblib"))
        except Exception:
            pass

        best_models[name] = final_model
        results.append({
            "dataset": name,
            "cv_strategy": cv_strategy,
            "cv_best_value": study.best_value,
            "test_balanced_accuracy": metrics.get("balanced_accuracy"),
            "test_roc_auc": metrics.get("roc_auc"),
            "test_mcc": metrics.get("mcc"),
            "test_sensitivity": metrics.get("sensitivity"),
            "test_specificity": metrics.get("specificity"),
            "n_trials": len(study.trials),
            "train_size": len(X_train),
            "test_size": len(X_test),
        })

    summary_df = pd.DataFrame(results).sort_values(by="test_balanced_accuracy", ascending=False).reset_index(drop=True)
    summary_df.to_csv(os.path.join(output_dir, "summary.csv"), index=False)
    
    print(f"\n{'='*60}")
    print("All datasets processed. Summary:")
    print(f"{'='*60}")
    print(summary_df)
    
    return summary_df, best_models, studies


# MCI_LBD vs HC -> OPTUNA + XGB

# From this one

In [ ]:
task_hf_compensated_comb_emb_lbl_lbd_hc.get("1_1")

In [ ]:
task_hf_compensated_comb_emb_lbl_lbd_hc.get("1_1")

In [ ]:
task_hf_comb_emb_lbl_lbd_hc.get("1_1")

In [ ]:
datasets = task_hf_compensated_comb_emb_lbl_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_lbd_hc/xgb_results_handcrafted_combined_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:
task_hf_compensated_comb_emb_lbl_lbd_hc.get("1_1")

In [ ]:
task_hf_compensated_comb_emb_lbl_lbd_hc.get('1_1')

In [ ]:

datasets = task_hf_compensated_temp_emb_lbl_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_lbd_hc/xgb_results_handcrafted_temporal_emb_compensated",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:
task_hf_compensated_temp_emb_lbl_lbd_hc.get('18_1').head(15)

In [ ]:

datasets = task_hf_compensated_freq_emb_lbl_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_lbd_hc/xgb_results_handcrafted_frerq_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)
## MCI-AD vs HC -> OPTUNA + XGB

In [ ]:

datasets = task_freq_dfs_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_lbd_hc/xgb_results_freq_lbd_hc_extended",
    split_param="emb_",
)
print(summary)


In [ ]:
task_freq_dfs_lbd_hc.get("1_1")

In [ ]:

# Temporal embedings
datasets = task_temp_dfs_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="balanced_accuracy",      
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_lbd_hc/xgb_results_temporal_extended",
    split_param="emb_",
)
print(summary)


In [ ]:
#TODO: Contains 2s in diagnosis
# Combined embedings
datasets = task_comb_dfs_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    n_trials=200,
    cv_strategy="skf",
    split_param="emb_",
)
print(summary)



In [ ]:

datasets = task_hf_dfs_clean_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_lbd_hc/xgb_results_handcrafted_extended",
    split_param="w.cz.fnusa",
)
print(summary)

## MCI-AD vs HC -> OPTUNA + XGB

In [ ]:

datasets = task_hf_compensated_comb_emb_lbl_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_hc/xgb_results_handcrafted_combined_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:
task_hf_compensated_temp_emb_lbl_ad_hc.get('9_1')

In [ ]:

datasets = task_hf_compensated_temp_emb_lbl_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_hc/xgb_results_handcrafted_temporal_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:

datasets = task_hf_compensated_freq_emb_lbl_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_hc/xgb_results_handcrafted_frerq_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)
## MCI-AD vs HC -> OPTUNA + XGB

In [ ]:
datasets = task_freq_dfs_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_hc/xgb_results_freq_ad_hc_extended",
    split_param="emb_",
)
print(summary)


In [ ]:

# Temporal embedings
datasets = task_temp_dfs_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="balanced_accuracy",      
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_hc/xgb_results_temporal_extended",
    split_param="emb_",
)
print(summary)


In [ ]:

# Combined embedings
datasets = task_comb_dfs_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_hc/xgb_results_combined_extended",
    split_param="emb_",
)
print(summary)



In [ ]:

datasets = task_hf_dfs_clean_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_hc/xgb_results_handcrafted_extended",
    split_param="w.cz.fnusa",
)
print(summary)

## MCI-LBD vs MCI-AD -> OTPUNA + XBG

In [ ]:
task_hf_compensated_comb_emb_lbl_ad_lbd.get("1_1")

In [ ]:
#TODO: contains label
datasets = task_hf_compensated_comb_emb_lbl_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_combined_emb_compensated_ad_lbd_extended",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:

datasets = task_hf_compensated_temp_emb_lbl_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_temporal_emb_compensated_ad_lbd_extended",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:

datasets = task_hf_compensated_freq_emb_lbl_ad_lbl

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_frerq_emb_compensated_ad_lbd_extended",
    split_param= "diagnosis_y",
)
print(summary)
## MCI-AD vs HC -> OPTUNA + XGB

In [ ]:
datasets = task_freq_dfs_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_freq_ad_lbd_extended",
    split_param="emb_",
)
print(summary)


In [ ]:

# Temporal embedings
datasets = task_temp_dfs_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="balanced_accuracy",      
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_temporal_ad_lbd_extended",
    split_param="emb_",
)
print(summary)


In [ ]:

# Combined embedings
datasets = task_comb_dfs_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_combined_ad_lbd_extended",
    split_param="emb_",
)
print(summary)



In [ ]:

datasets = task_hf_dfs_clean_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_ad_lbd_extended",
    split_param="w.cz.fnusa",
)
print(summary)

## MCI-PD vs HC -> OPTUNA + XGB

In [ ]:
#TODO: [0,4] in diagnosis
datasets = task_hf_compensated_comb_emb_lbl_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_pd_hc/xgb_results_handcrafted_combined_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:

datasets = task_hf_compensated_temp_emb_lbl_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_pd_hc/xgb_results_handcrafted_temporal_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:

datasets = task_hf_compensated_freq_emb_lbl_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_pd_hc/xgb_results_handcrafted_frerq_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)
## MCI-AD vs HC -> OPTUNA + XGB

In [ ]:
datasets = task_freq_dfs_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_pd_hc/xgb_results_freq_pd_hc_extended",
    split_param="emb_",
)
print(summary)


In [ ]:

# Temporal embedings
datasets = task_temp_dfs_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="balanced_accuracy",      
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_hc/xgb_results_temporal_extended",
    split_param="emb_",
)
print(summary)


In [ ]:

# Combined embedings
datasets = task_comb_dfs_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_hc/xgb_results_combined_extended",
    split_param="emb_",
)
print(summary)



In [ ]:

datasets = task_hf_dfs_clean_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_hc/xgb_results_handcrafted_extended",
    split_param="w.cz.fnusa",
)
print(summary)

# MCI-LBD vs HC - 9_1 - Long Sentence

In [874]:
task_9_1_comb_emb_hf_lbd_hc = {"9_1": task_hf_compensated_comb_emb_lbl_lbd_hc.get("9_1")}
task_9_1_temp_emb_hf_lbd_hc = {"9_1": task_hf_compensated_temp_emb_lbl_lbd_hc.get("9_1")}
task_9_1_freq_emb_hf_lbd_hc = {"9_1": task_hf_compensated_freq_emb_lbl_lbd_hc.get("9_1")}

task_9_1_comb_emb_hf_lbd_hc = {"9_1": task_comb_dfs_lbd_hc.get("9_1")}
task_9_1_temp_emb_hf_lbd_hc = {"9_1": task_temp_dfs_lbd_hc.get("9_1")}
task_9_1_freq_emb_hf_lbd_hc = {"9_1": task_freq_dfs_lbd_hc.get("9_1")}

task_9_1_freq_emb_hf_lbd_hc = {"9_1": task_hf_dfs_clean_lbd_hc.get("9_1")}

In [871]:
print(task_9_1_comb_emb_hf_lbd_hc)

{'9_1':                     subject     emb_0     emb_1     emb_2     emb_3     emb_4  \
0    COBEN-WTABLET-AS-HCD05 -0.025408 -0.001727 -0.042265  0.082035 -0.035739   
1     COBEN-WTABLET-DS-HC36 -0.024942  0.000568 -0.038449  0.085552 -0.031823   
2     COBEN-WTABLET-EH-HC35 -0.027187  0.003978 -0.027247  0.077886 -0.025291   
3     COBEN-WTABLET-FG-HC38 -0.027634 -0.000744 -0.035132  0.081841 -0.026633   
4     COBEN-WTABLET-HJ-HC21 -0.028347  0.001857 -0.046883  0.083798 -0.051095   
..                      ...       ...       ...       ...       ...       ...   
154            pre-LBD-55#1 -0.027970 -0.024335 -0.057045  0.006718  0.047345   
155            pre-LBD-56#1 -0.027627 -0.024618 -0.059288  0.003451  0.044747   
156            pre-LBD-57#1 -0.028777 -0.025344 -0.058944  0.006279  0.046631   
157            pre-LBD-58#1 -0.030584 -0.023180 -0.062114  0.001802  0.045685   
158            pre-LBD-59#1 -0.031710 -0.024203 -0.065047  0.006085  0.045745   

        emb_5     e

In [867]:
task_9_1_comb_emb_hf_lbd_hc

,subject,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,w.cz.fnusa.9_1_slope of velocity (on-surface),w.cz.fnusa.19_1_slope of vertical velocity (in-air),w.cz.fnusa.9_1_slope of vertical velocity (in-air),w.cz.fnusa.19_1_slope of vertical velocity (on-surface),w.cz.fnusa.9_1_slope of vertical velocity (on-surface),w.cz.fnusa.19_1_tempo (in-air),w.cz.fnusa.9_1_tempo (in-air),w.cz.fnusa.19_1_tempo (on-surface),w.cz.fnusa.9_1_tempo (on-surface),diagnosis_y
0,COBEN-WTABLET-AS-HCD05,-0.025408,-0.001727,-0.042265,0.082035,-0.035739,0.021134,0.001806,0.011006,0.023722,...,0.009227,0.006966,-0.007292,-0.001003,0.004135,3.020668,2.913025,1.249609,1.334401,0.0
1,COBEN-WTABLET-DS-HC36,-0.024942,0.000568,-0.038449,0.085552,-0.031823,0.019596,0.002355,0.012827,0.027737,...,0.008251,-0.021440,-0.004304,-0.002413,0.006475,3.143666,3.349282,1.281504,1.476959,0.0
2,COBEN-WTABLET-EH-HC35,-0.027187,0.003978,-0.027247,0.077886,-0.025291,0.010023,0.003236,0.005319,0.018954,...,0.008656,-0.004459,0.007066,0.000301,0.007160,2.909620,2.622860,1.786778,1.655081,0.0
3,COBEN-WTABLET-FG-HC38,-0.027634,-0.000744,-0.035132,0.081841,-0.026633,0.017900,0.005049,0.008424,0.018940,...,0.006053,-0.006564,0.008971,-0.004395,0.004306,1.870972,1.413604,1.785031,1.706485,0.0
4,COBEN-WTABLET-HJ-HC21,-0.028347,0.001857,-0.046883,0.083798,-0.051095,0.021357,-0.005586,0.013864,0.030507,...,-0.006043,-0.000683,-0.021849,-0.005806,-0.006378,1.783040,2.737995,1.852116,1.407177,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,pre-LBD-55#1,-0.027970,-0.024335,-0.057045,0.006718,0.047345,-0.004125,-0.036797,-0.038131,0.001233,...,0.016890,-0.015035,-0.035060,-0.006905,0.007775,3.053680,2.243829,1.692477,1.557632,1.0
155,pre-LBD-56#1,-0.027627,-0.024618,-0.059288,0.003451,0.044747,-0.004398,-0.035753,-0.038674,0.001671,...,0.007319,-0.002531,0.002703,-0.001846,0.004136,3.220612,3.925121,1.202008,1.479134,1.0
156,pre-LBD-57#1,-0.028777,-0.025344,-0.058944,0.006279,0.046631,-0.003365,-0.036401,-0.040138,0.001929,...,0.002162,-0.002900,-0.006378,-0.003869,-0.005568,3.606103,4.647999,6.519806,5.487327,1.0
157,pre-LBD-58#1,-0.030584,-0.023180,-0.062114,0.001802,0.045685,-0.005894,-0.034360,-0.039845,-0.003050,...,0.005370,0.009638,-0.019467,0.000220,0.006354,3.263052,3.200569,1.099937,1.135847,1.0


In [ ]:
datasets = task_9_1_comb_emb_hf_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    n_trials=20,
    cv_strategy="loo"
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_combined_emb_compensated_ad_lbd_extended",
    split_param= "diagnosis_y",
)
print(summary)


[I 2025-11-25 15:27:44,027] A new study created in memory with name: 9_1_balanced_accuracy



Processing dataset: 9_1
Train size: 127, Test size: 32
Using Stratified K-Fold CV with 5 folds
Starting optimization with 200 trials...


  0%|          | 0/200 [00:00<?, ?it/s]

In [ ]:

datasets = task_hf_compensated_temp_emb_lbl_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_temporal_emb_compensated_ad_lbd_extended",
    split_param= "diagnosis_y",
)
print(summary)


In [ ]:

datasets = task_hf_compensated_freq_emb_lbl_ad_lbl

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_frerq_emb_compensated_ad_lbd_extended",
    split_param= "diagnosis_y",
)
print(summary)
## MCI-AD vs HC -> OPTUNA + XGB

In [ ]:
datasets = task_freq_dfs_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_freq_ad_lbd_extended",
    split_param="emb_",
)
print(summary)


In [ ]:

# Temporal embedings
datasets = task_temp_dfs_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="balanced_accuracy",      
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_temporal_ad_lbd_extended",
    split_param="emb_",
)
print(summary)


In [ ]:

# Combined embedings
datasets = task_comb_dfs_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_combined_ad_lbd_extended",
    split_param="emb_",
)
print(summary)



In [ ]:

datasets = task_hf_dfs_clean_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    cv_strategy="skf",
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_ad_lbd_extended",
    split_param="w.cz.fnusa",
)
print(summary)